# nb6 — Phase 2 · Bước 2: Post-processing chống deletion (eval-only · CPU)

Notebook thứ sáu (`DESIGN.md` §12, quyết định 24/09): dựng **rule tất định gold-agnostic** hồi phục FP deletion — nhóm FP lớn nhất của Run 2 trên test (**45,4%** tổng FP; trần lý thuyết **+2,33 F1** nếu chặn hết delete-FP — `REPORT.md` §7.2/§9).

**Nguyên tắc chống leakage (`DESIGN.md` §9 — bắt buộc):**
- Toàn bộ pattern/whitelist/blacklist derive **CHỈ từ val Run 2** (nguồn duy nhất của rule) — findings nb4 trên test chỉ dùng kiểm chứng chiều hướng sau khi freeze, không tạo rule từ test.
- Sweep grid giới hạn (**≤6 tổ hợp lồng nhau**) trên val n=927; ngưỡng nhận config: **cải thiện ≥ 0,3 điểm F1 trên val** (`MIN_VAL_GAIN = 0.003`) + over-correction không tăng.
- Config frozen áp lên test (Run 1 + Run 2); **Run 1 test = robustness check** của config.

**Eval-only · CPU**: không model, không GPU, không pip install — chỉ stdlib + file prediction có sẵn của nb3 (self-contained: `row_id/text/corrected_text/prediction`).

**Input** (Add Input trên Kaggle, hoặc đặt vào `./out` local):
- `predictions_val_run{1,2}.jsonl`, `predictions_test_run{1,2}.jsonl` — nb3.
- `test_aligned.jsonl` — nb1 (đối chiếu text của 2 file test).
- `syllable_table.json` — nb2 (7.884 âm tiết NFC + lower — cho R2).

**Output**: `postprocess_config.json` (frozen — nb7 nạp để đối chiếu), `postprocess_report.json`, `predictions_{val,test}_run{1,2}_postproc.jsonl`.

## Nội dung
0. Cấu hình + tự dò input
1. Cell hàm dùng chung (copy nguyên vẹn từ nb5 — `SHARED_CELLS_VERSION = 'align-v1'`) + `evaluate_predictions`
2. Mining FP + rule engine (R1–R4) + sanity nhân tạo
3. FP mining trên **val Run 2** — nguồn duy nhất của rule
4. Sweep grid trên val Run 2 → chọn config
5. Frozen áp lên val (báo cáo) + test Run 1/Run 2 — Δ4 chỉ số, hit-count, đối chiếu trần +2,33
6. QA 10 mẫu + kiểm định NFC
7. Xuất file

## 0. Cấu hình & tự dò input

Mọi tham số gom một chỗ (`DESIGN.md` §7). Không nạp torch/model — notebook chạy thuần CPU.

In [1]:
import json
import datetime
import unicodedata
from pathlib import Path
import collections

# --- Cấu hình nb6 (mọi tham số gom một chỗ — DESIGN.md §7) ---
SEED = 42
MIN_VAL_GAIN = 0.003        # ngưỡng nhận config post-processing: >= 0,3 điểm F1 trên val
R3_K_GRID = [1, 3, 5]       # grid whitelist frequency k (dùng trong GRID dưới)
PHRASE_MIN_COUNT = 2        # phrase vào blacklist R4 nếu lặp >= 2 lần trong FP list val
THEORETICAL_CEILING_F1_GAIN = 0.0233  # trần lý thuyết test Run 2 nếu chặn hết delete-FP (nb4) — chỉ đối chiếu

REQUIRED_FILES = [
    'predictions_val_run1.jsonl',    # nb3
    'predictions_val_run2.jsonl',    # nb3 — tập TUNE (nguồn duy nhất của rule)
    'predictions_test_run1.jsonl',   # nb3 — robustness check
    'predictions_test_run2.jsonl',   # nb3 — tập APPLY (frozen)
    'test_aligned.jsonl',            # nb1 — đối chiếu text của 2 file test
    'syllable_table.json',           # nb2 — bảng âm tiết cho R2
]


def find_required_inputs():
    candidates = []
    kaggle = Path('/kaggle/input')
    if kaggle.is_dir():
        candidates += sorted(kaggle.rglob('*.jsonl')) + sorted(kaggle.rglob('*.json'))
    local = Path('./out')
    if local.is_dir():
        candidates += sorted(local.rglob('*.jsonl')) + sorted(local.rglob('*.json'))
    found = {}
    for p in candidates:
        if p.name in REQUIRED_FILES and p.name not in found:
            found[p.name] = p
    missing = [req for req in REQUIRED_FILES if req not in found]
    if missing:
        raise FileNotFoundError(
            'Thiếu input bắt buộc: ' + ', '.join(missing) +
            ' | nb6 cần Add Input: output nb3 (4 file predictions_{val,test}_run{1,2}.jsonl), '
            'output nb1 (test_aligned.jsonl), output nb2 (syllable_table.json). '
            'Local: đặt file vào ./out.'
        )
    return found


INPUT_FILES = find_required_inputs()

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('./out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STAMP = datetime.datetime.now().isoformat(timespec='seconds')

SYLL_TABLE = json.loads(Path(INPUT_FILES['syllable_table.json']).read_text(encoding='utf-8'))
SYLL_SET = set(SYLL_TABLE['entries'])

print('=== TỰ DÒ INPUT HOÀN TẤT (eval-only · CPU — không nạp model) ===')
for k in REQUIRED_FILES:
    print(f'  {k:32s}: {INPUT_FILES[k]}')
print(f'Bảng âm tiết: {len(SYLL_SET)} entries · OUTPUT_DIR: {OUTPUT_DIR}')

=== TỰ DÒ INPUT HOÀN TẤT (eval-only · CPU — không nạp model) ===
  predictions_val_run1.jsonl      : /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/predictions_val_run1.jsonl
  predictions_val_run2.jsonl      : /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/predictions_val_run2.jsonl
  predictions_test_run1.jsonl     : /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/predictions_test_run1.jsonl
  predictions_test_run2.jsonl     : /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/predictions_test_run2.jsonl
  test_aligned.jsonl              : /kaggle/input/notebooks/cquangnguynl/nb1-align-annotate/test_aligned.jsonl
  syllable_table.json             : /kaggle/input/notebooks/cquangnguynl/nb2-pilot-dict-noise/syllable_table.json
Bảng âm tiết: 7884 entries · OUTPUT_DIR: /kaggle/working


## 1. Cell hàm dùng chung — copy NGUYÊN VẸT từ nb5 (nguồn đầy đủ nhất của `align-v1`)

Bản shared cell của nb5 đã hợp nhất cả 2 nhánh: nb1/nb2 (`build_pseudo_annotation`, `SUSPECT_EDIT_RATIO`) và nb3 (`levenshtein_opcodes` → `apply_edit_blocks`, `load_jsonl`). Copy nguyên vẹn không sửa một ký tự (nguyên tắc `DESIGN.md` §7: hàm dùng chung bị sửa ở ≥2 nơi thì phải trích `.py` chung).

In [2]:
import re

SHARED_CELLS_VERSION = 'align-v1'

_WS_RE = re.compile(r'\s+')
_TOKEN_RE = re.compile(r'\w+|[^\w\s]+')
_HAS_WORD_RE = re.compile(r'\w')
_DIGIT_RE = re.compile(r'\d')

def nfc_normalize(s):
    return _WS_RE.sub(' ', unicodedata.normalize('NFC', s)).strip()

def canon_tokenize(s):
    """NFC + tách token: run chữ/số liền kề (\\w+) là 1 token; cụm dấu câu liền kề là 1 token riêng."""
    return _TOKEN_RE.findall(nfc_normalize(s))

def is_punct_token(tok):
    return not _HAS_WORD_RE.search(tok)

def is_word_token(tok):
    return not is_punct_token(tok) and not _DIGIT_RE.search(tok)

def levenshtein_opcodes(src, tgt):
    n, m = len(src), len(tgt)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        dp[i][0] = i
    for j in range(1, m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        si = src[i - 1]
        prev, row = dp[i - 1], dp[i]
        for j in range(1, m + 1):
            best = prev[j - 1] + (0 if si == tgt[j - 1] else 1)
            if prev[j] + 1 < best:
                best = prev[j] + 1
            if row[j - 1] + 1 < best:
                best = row[j - 1] + 1
            row[j] = best
    ops = []
    def _push(tag, i1, i2, j1, j2):
        if ops and ops[-1][0] == tag and ops[-1][1] == i2 and ops[-1][3] == j2:
            ops[-1] = (tag, i1, ops[-1][2], j1, ops[-1][4])
        else:
            ops.append((tag, i1, i2, j1, j2))
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + (0 if src[i - 1] == tgt[j - 1] else 1):
            _push('equal' if src[i - 1] == tgt[j - 1] else 'replace', i - 1, i, j - 1, j)
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            _push('delete', i - 1, i, j, j)
            i -= 1
        else:
            _push('insert', i, i, j - 1, j)
            j -= 1
    ops.reverse()
    return ops

def _make_block(src, tgt, span):
    i1, i2, j1, j2 = span
    src_toks, tgt_toks = src[i1:i2], tgt[j1:j2]
    ns, nt = len(src_toks), len(tgt_toks)
    if ns == 0:
        btype = 'insert'
    elif nt == 0:
        btype = 'delete'
    elif ns == 1 and nt == 1:
        btype = 'substitute'
    elif ns == 1:
        btype = 'split'
    elif nt == 1:
        btype = 'merge'
    else:
        btype = 'multi'
    return {
        'type': btype,
        'position': i1,
        'src_span': [i1, i2],
        'tgt_span': [j1, j2],
        'src_tokens': src_toks,
        'tgt_tokens': tgt_toks,
        'punct_only': all(is_punct_token(t) for t in src_toks + tgt_toks),
    }

def extract_edit_blocks(src, tgt, opcodes):
    blocks, cur = [], None
    for tag, i1, i2, j1, j2 in opcodes:
        if tag == 'equal':
            if cur is not None:
                blocks.append(_make_block(src, tgt, cur))
                cur = None
        elif cur is None:
            cur = [i1, i2, j1, j2]
        else:
            cur[1], cur[3] = i2, j2
    if cur is not None:
        blocks.append(_make_block(src, tgt, cur))
    return blocks

def apply_edit_blocks(src, blocks):
    out, pos = [], 0
    for b in blocks:
        out += src[pos:b['src_span'][0]]
        out += b['tgt_tokens']
        pos = b['src_span'][1]
    out += src[pos:]
    return out

SUSPECT_EDIT_RATIO = 0.3  # giữ nguyên giá trị nb1/nb2 — cell hàm dùng chung copy nguyên vẹn cần nó

def build_pseudo_annotation(text, corrected, suspect_ratio=SUSPECT_EDIT_RATIO):
    """Align text ↔ corrected_text trong hệ token canonical → pseudo-annotation schema giống VSEC + edit_blocks.
    Quy ước QUAN TRỌNG (các notebook sau phải dùng cell này nguyên vẹn):
    - error_positions: index token nguồn nằm trong src_span của block CÓ token nguồn.
      Block insert (thiếu âm tiết ở nguồn) KHÔNG có vị trí nguồn → không nằm trong error_positions,
      chỉ nằm trong correction_pairs với error='' và position = vị trí chèn trước (có thể == len(src)).
    - correction_pairs: 1 entry/block; error/correction = các token nối bằng space; delete → correction=''.
    - syllable_annotations: 1 entry/token nguồn; is_correct=False khi token thuộc src_span của block
      non-insert; corrections = chuỗi đích (join space) của block đó.
    - suspect: edit_ratio = (tổng token cả 2 vế nằm trong edit block) / max(len(src), len(tgt)) vượt ngưỡng.
    - align_failed: một trong hai vế token hóa rỗng."""
    src = canon_tokenize(text)
    tgt = canon_tokenize(corrected)
    blocks = extract_edit_blocks(src, tgt, levenshtein_opcodes(src, tgt))
    corrections_by_pos = {}
    for b in blocks:
        if b['src_span'][0] < b['src_span'][1]:
            fix = ' '.join(b['tgt_tokens'])
            for i in range(b['src_span'][0], b['src_span'][1]):
                corrections_by_pos.setdefault(i, []).append(fix)
    error_positions = sorted(corrections_by_pos)
    syllable_annotations = [
        {
            'syllable': tok,
            'is_correct': i not in corrections_by_pos,
            'corrections': corrections_by_pos.get(i, []),
            'position': i,
        }
        for i, tok in enumerate(src)
    ]
    correction_pairs = [
        {'error': ' '.join(b['src_tokens']), 'correction': ' '.join(b['tgt_tokens']), 'position': b['position']}
        for b in blocks
    ]
    n_edit_tokens = sum(
        (b['src_span'][1] - b['src_span'][0]) + (b['tgt_span'][1] - b['tgt_span'][0]) for b in blocks
    )
    denom = max(len(src), len(tgt))
    edit_ratio = n_edit_tokens / denom if denom else 0.0
    return {
        'is_clean': not blocks,
        'align_failed': not src or not tgt,
        'suspect': edit_ratio > suspect_ratio,
        'edit_ratio': round(edit_ratio, 4),
        'error_count': len(blocks),
        'error_positions': error_positions,
        'correction_pairs': correction_pairs,
        'syllable_annotations': syllable_annotations,
        'edit_blocks': blocks,
        'src_tokens': src,
        'tgt_tokens': tgt,
    }

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

print('SHARED_CELLS_VERSION:', SHARED_CELLS_VERSION)

SHARED_CELLS_VERSION: align-v1


### Hàm đánh giá chuẩn 3 lớp chỉ số — copy nguyên vẹn từ nb5 (giống nb3 §1)

`evaluate_predictions` chỉ yêu cầu record có khóa `text` + `corrected_text` — file prediction jsonl của nb3 dùng trực tiếp làm records, không cần file gốc. Sanity case nhân tạo giữ nguyên.

In [3]:
def evaluate_predictions(records, predictions, syll_set=None):
    assert len(records) == len(predictions), f'Độ dài không khớp: {len(records)} vs {len(predictions)}'
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    correct_at_tp = 0
    
    total_clean_tokens = 0
    clean_sents_total = 0
    clean_sents_preserved = 0
    
    # Stratified stats: non-word vs real-word
    stratified = {
        'nonword': {'gold': 0, 'detected': 0, 'corrected': 0},
        'realword': {'gold': 0, 'detected': 0, 'corrected': 0}
    }
    
    sample_overcorrections = []
    
    for rec, pred in zip(records, predictions):
        src_toks = canon_tokenize(rec['text'])
        gold_toks = canon_tokenize(rec['corrected_text'])
        pred_toks = canon_tokenize(pred)
        
        # Gold edit blocks
        gold_ops = levenshtein_opcodes(src_toks, gold_toks)
        gold_blocks = extract_edit_blocks(src_toks, gold_toks, gold_ops)
        
        # Pred edit blocks
        pred_ops = levenshtein_opcodes(src_toks, pred_toks)
        pred_blocks = extract_edit_blocks(src_toks, pred_toks, pred_ops)
        
        gold_pos_map = {}
        for b in gold_blocks:
            if b['src_span'][0] < b['src_span'][1]:
                target_str = ' '.join(b['tgt_tokens'])
                for p in range(b['src_span'][0], b['src_span'][1]):
                    gold_pos_map[p] = (target_str, b['src_tokens'])
                    
        pred_pos_map = {}
        for b in pred_blocks:
            if b['src_span'][0] < b['src_span'][1]:
                pred_str = ' '.join(b['tgt_tokens'])
                for p in range(b['src_span'][0], b['src_span'][1]):
                    pred_pos_map[p] = (pred_str, b['src_tokens'])
                    
        gold_positions = set(gold_pos_map.keys())
        pred_positions = set(pred_pos_map.keys())
        
        tp_pos = gold_positions & pred_positions
        fp_pos = pred_positions - gold_positions
        fn_pos = gold_positions - pred_positions
        
        total_tp += len(tp_pos)
        total_fp += len(fp_pos)
        total_fn += len(fn_pos)
        
        # Đếm số token đúng trong câu nguồn (loại bỏ token dấu câu)
        word_token_positions = {i for i, t in enumerate(src_toks) if is_word_token(t)}
        clean_word_positions = word_token_positions - gold_positions
        total_clean_tokens += len(clean_word_positions)
        
        # Check clean sentence
        if len(gold_positions) == 0:
            clean_sents_total += 1
            if len(pred_positions) == 0:
                clean_sents_preserved += 1
                
        # Correction accuracy
        for p in tp_pos:
            target_str, _ = gold_pos_map[p]
            pred_str, _ = pred_pos_map[p]
            if pred_str.lower() == target_str.lower():
                correct_at_tp += 1
                
            # Stratified
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['detected'] += 1
                if pred_str.lower() == target_str.lower():
                    stratified[cat]['corrected'] += 1
                    
        for p in fn_pos:
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['gold'] += 1
        for p in tp_pos:
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['gold'] += 1
                
        # Lưu mẫu over-correction tiêu biểu
        if fp_pos and len(sample_overcorrections) < 20:
            for p in sorted(fp_pos):
                if p < len(src_toks):
                    sample_overcorrections.append({
                        'orig_token': src_toks[p],
                        'pred_token': pred_pos_map[p][0],
                        'context': ' '.join(src_toks[max(0, p-3):min(len(src_toks), p+4)]),
                        'full_src': rec['text'],
                        'full_pred': pred
                    })
                    if len(sample_overcorrections) >= 20:
                        break

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    corr_acc = correct_at_tp / total_tp if total_tp > 0 else 0.0
    overcorr_rate = total_fp / total_clean_tokens if total_clean_tokens > 0 else 0.0
    clean_retention = clean_sents_preserved / clean_sents_total if clean_sents_total > 0 else 1.0

    return {
        'detection': {
            'tp': total_tp,
            'fp': total_fp,
            'fn': total_fn,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        },
        'correction': {
            'correct_at_tp': correct_at_tp,
            'accuracy': corr_acc,
        },
        'over_correction': {
            'fp_count': total_fp,
            'clean_tokens': total_clean_tokens,
            'rate': overcorr_rate,
            'clean_sents_total': clean_sents_total,
            'clean_sents_preserved': clean_sents_preserved,
            'clean_retention_rate': clean_retention,
        },
        'stratified': stratified,
        'sample_overcorrections': sample_overcorrections,
    }

# Sanity Test hàm đánh giá trên ví dụ giả định (copy nguyên vẹn từ nb3)
_test_recs = [{'text': 'học sanh đi hoc', 'corrected_text': 'học sinh đi học'}]
_test_preds = ['học sinh đi hoc'] # Model sửa 'sanh' -> 'sinh', nhưng bỏ sót 'hoc'
_m_test = evaluate_predictions(_test_recs, _test_preds)
assert _m_test['detection']['tp'] == 1 and _m_test['detection']['fn'] == 1 and _m_test['detection']['fp'] == 0
assert _m_test['correction']['accuracy'] == 1.0
print('Sanity check hàm evaluate_predictions PASS 100%!')

Sanity check hàm evaluate_predictions PASS 100%!


## 2. Mining FP + rule engine (R1–R4)

**Cơ chế hoàn nguyên** (cùng cấp độ khối chỉnh sửa với align-v1): align `canon_tokenize(src) ↔ canon_tokenize(pred)` qua `levenshtein_opcodes` + `extract_edit_blocks`; block bị veto → thay `tgt_tokens` bằng `src_tokens`; rebuild bằng `apply_edit_blocks`. Không block nào bị veto → trả về **prediction gốc nguyên vẹn** (identity guarantee).

**Mining FP (gold-aware — chỉ chạy trên val):** block của pred mà **không** vị trí nguồn nào nằm trong gold positions = pure-FP block. Block mixed (một phần vị trí là TP) **không thể** hoàn nguyên nguyên khối → chỉ thống kê, không tạo rule.

| Rule | Điều kiện veto (gold-agnostic khi áp) | Grid |
|---|---|---|
| R1 punctuation guard | block `punct_only` | on/off |
| R2 veto-delete âm tiết hợp lệ | block thuần xóa (tgt rỗng) mà **mọi** token bị xóa có `.lower()` ∈ bảng âm tiết | on/off |
| R3 whitelist val | block thuần xóa mà mọi token bị xóa có tần suất ≥ k trong FP-deletion list của **val** | k ∈ {1,3,5} |
| R4 phrase blacklist val | block ≥2 token nguồn, phrase (join lower) nằm trong FP-phrase list của **val** (lặp ≥ `PHRASE_MIN_COUNT`) | on/off |

FP substitution 1-token (vd `'Nguwo'` ×95 của nb4 trên test) nằm ngoài R1–R4 → ghi nhận residual qua phân bố type, không xử lý lần này.

In [4]:
def load_prediction_set(path):
    recs = load_jsonl(path)
    preds = [r['prediction'] for r in recs]
    assert len(recs) == len(preds)
    for r in recs:
        assert 'text' in r and 'corrected_text' in r and 'prediction' in r, r.get('row_id')
    return recs, preds


def mine_fp_blocks(records, preds):
    """Đếm FP theo edit block trên 1 prediction set (gold-aware — CHỈ dùng cho val, chống leakage DESIGN.md §9).
    Pure-FP block = block của pred mà không vị trí nguồn nào nằm trong gold positions.
    Block mixed (một phần vị trí là TP) không thể hoàn nguyên nguyên khối → chỉ thống kê.
    Trả về: deleted_tokens (block thuần xóa), fp_phrases (block >=2 token), punct_tokens (block punct-only), stats."""
    deleted_tokens = collections.Counter()
    fp_phrases = collections.Counter()
    punct_tokens = collections.Counter()
    stats = collections.Counter()
    for rec, pred in zip(records, preds):
        src = canon_tokenize(rec['text'])
        gold = canon_tokenize(rec['corrected_text'])
        ptoks = canon_tokenize(pred)
        gold_blocks = extract_edit_blocks(src, gold, levenshtein_opcodes(src, gold))
        gold_pos = {p for b in gold_blocks if b['src_span'][0] < b['src_span'][1]
                    for p in range(b['src_span'][0], b['src_span'][1])}
        pred_blocks = extract_edit_blocks(src, ptoks, levenshtein_opcodes(src, ptoks))
        for b in pred_blocks:
            if b['src_span'][0] >= b['src_span'][1]:
                continue  # insert block: không có vị trí nguồn để đối chiếu
            positions = set(range(b['src_span'][0], b['src_span'][1]))
            if positions & gold_pos:
                stats['mixed_blocks_not_revertible'] += 1
                continue
            stats['fp_blocks'] += 1
            stats['fp_type_' + b['type']] += 1
            if b['punct_only']:
                stats['fp_punct_only'] += 1
                for t in b['src_tokens']:
                    punct_tokens[t] += 1
            if b['type'] == 'delete':
                for t in b['src_tokens']:
                    deleted_tokens[nfc_normalize(t).lower()] += 1
            if len(b['src_tokens']) >= 2:
                fp_phrases[' '.join(nfc_normalize(t).lower() for t in b['src_tokens'])] += 1
    return {'deleted_tokens': deleted_tokens, 'fp_phrases': fp_phrases,
            'punct_tokens': punct_tokens, 'stats': stats}

In [5]:
def build_rule_resources(mining, k, phrase_min_count=PHRASE_MIN_COUNT):
    """Resource R3/R4 derive từ mining val (nguồn duy nhất của rule — không derive từ test)."""
    return {
        'r3_whitelist': {t for t, n in mining['deleted_tokens'].items() if n >= k} if k > 0 else set(),
        'r4_phrases': {p for p, n in mining['fp_phrases'].items() if n >= phrase_min_count},
    }


def veto_reason(block, cfg, res, syll_set):
    """Trả về tên rule nếu block bị phủ quyết (gold-agnostic khi áp: chỉ nhìn block shape + resource từ val)."""
    if block['src_span'][0] >= block['src_span'][1]:
        return None  # insert block: không hoàn nguyên được
    if cfg.get('r1_punct_guard') and block['punct_only']:
        return 'R1'
    is_pure_delete = block['tgt_span'][0] == block['tgt_span'][1]
    if is_pure_delete:
        low_toks = [nfc_normalize(t).lower() for t in block['src_tokens']]
        if cfg.get('r2_veto_valid_syllable') and low_toks and all(t in syll_set for t in low_toks):
            return 'R2'
        if cfg.get('r3_whitelist_min_count', 0) > 0 and low_toks \
                and all(t in res['r3_whitelist'] for t in low_toks):
            return 'R3'
    if cfg.get('r4_phrase_blacklist') and len(block['src_tokens']) >= 2:
        phrase = ' '.join(nfc_normalize(t).lower() for t in block['src_tokens'])
        if phrase in res['r4_phrases']:
            return 'R4'
    return None


def postprocess_one(text, pred, cfg, res, syll_set):
    """Áp rule lên 1 cặp (text, pred): veto block → hoàn nguyên tgt = src, rebuild bằng apply_edit_blocks.
    Không block nào bị veto → trả về prediction gốc NGUYÊN VẸN (identity guarantee)."""
    src = canon_tokenize(text)
    ptoks = canon_tokenize(pred)
    blocks = extract_edit_blocks(src, ptoks, levenshtein_opcodes(src, ptoks))
    hits = collections.Counter()
    changed = False
    for b in blocks:
        reason = veto_reason(b, cfg, res, syll_set)
        if reason:
            b['tgt_tokens'] = list(b['src_tokens'])
            b['tgt_span'] = list(b['src_span'])
            hits[reason] += 1
            changed = True
    if not changed:
        return pred, hits
    out_toks = apply_edit_blocks(src, blocks)
    out = ' '.join(out_toks)
    assert canon_tokenize(out) == out_toks, pred          # roundtrip token hóa
    assert unicodedata.is_normalized('NFC', out), pred    # NFC invariant
    return out, hits


def postprocess_preds(records, preds, cfg, res, syll_set):
    outs, hit_counts = [], collections.Counter()
    n_changed = 0
    for rec, pred in zip(records, preds):
        out, hits = postprocess_one(rec['text'], pred, cfg, res, syll_set)
        outs.append(out)
        if out != pred:
            n_changed += 1
        hit_counts.update(hits)
    return outs, {'n_changed_sentences': n_changed, 'veto_by_rule': dict(hit_counts)}


# --- Sanity case nhân tạo (không dùng dữ liệu thật) ---
_syll = {'hòa', 'học', 'sinh', 'an', 'tòa'}
_res1 = {'r3_whitelist': {'xyzq'}, 'r4_phrases': {'biểm họa'}}
_cfg_r12 = {'r1_punct_guard': True, 'r2_veto_valid_syllable': True,
            'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': False}

# identity prediction → nguyên vẹn
_t, _h = postprocess_one('học sinh hòa an', 'học sinh hòa an', _cfg_r12, _res1, _syll)
assert _t == 'học sinh hòa an' and not _h
# R2: xóa 'hòa' (âm tiết hợp lệ) → hoàn tác
_t, _h = postprocess_one('học sinh hòa an', 'học sinh an', _cfg_r12, _res1, _syll)
assert _t == 'học sinh hòa an' and _h['R2'] == 1, (_t, _h)
# R2 off → giữ nguyên xóa
_cfg_no = dict(_cfg_r12, r2_veto_valid_syllable=False)
_t, _h = postprocess_one('học sinh hòa an', 'học sinh an', _cfg_no, _res1, _syll)
assert _t == 'học sinh an' and not _h
# xóa non-word ngoài whitelist → R2 KHÔNG bắn (không hoàn tác oan), giữ xóa
_t, _h = postprocess_one('học xyzq an', 'học an', _cfg_r12, _res1, _syll)
assert _t == 'học an' and not _h
# R3: token trong whitelist (k=1) → hoàn tác
_cfg_r13 = dict(_cfg_r12, r3_whitelist_min_count=1)
_t, _h = postprocess_one('học xyzq an', 'học an', _cfg_r13, _res1, _syll)
assert _t == 'học xyzq an' and _h['R3'] == 1, (_t, _h)
# grid k=0 (không whitelist) → giữ xóa
_cfg_r2only = {'r1_punct_guard': False, 'r2_veto_valid_syllable': False,
               'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': False}
_t, _h = postprocess_one('học xyzq an', 'học an', _cfg_r2only, _res1, _syll)
assert _t == 'học an' and not _h
# R1: xóa dấu câu → hoàn tác
_t, _h = postprocess_one('học sinh , an', 'học sinh an', _cfg_r12, _res1, _syll)
assert _t == 'học sinh , an' and _h['R1'] == 1, (_t, _h)
# R4: phrase blacklist 2-token → hoàn tác cả block
_cfg_r4 = {'r1_punct_guard': False, 'r2_veto_valid_syllable': False,
           'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': True}
_res4 = {'r3_whitelist': set(), 'r4_phrases': {'biểm họa'}}
_t, _h = postprocess_one('biểm họa lan', 'lan', _cfg_r4, _res4, _syll)
assert _t == 'biểm họa lan' and _h['R4'] == 1, (_t, _h)
print('Sanity rule engine PASS (identity · R1 · R2 on/off · R2 miss non-word · R3 whitelist · R4 phrase)')

Sanity rule engine PASS (identity · R1 · R2 on/off · R2 miss non-word · R3 whitelist · R4 phrase)


## 3. FP mining trên VAL Run 2 — nguồn duy nhất của rule (chống leakage)

Đếm trên val Run 2: (a) token bị model xóa tại vị trí FP (top list + tần suất → R3); (b) phrase FP lặp ≥2 token (→ R4); (c) FP trên block punct-only (R1); kèm phân bố type block FP — đối chiếu **chiều hướng** với nb4 test (45,4% delete) chỉ để kiểm chứng, không derive từ test.

In [6]:
PRED_SETS = {name: load_prediction_set(INPUT_FILES[f'predictions_{name}.jsonl'])
             for name in ['val_run1', 'val_run2', 'test_run1', 'test_run2']}

# Đối chiếu text 2 file test với test_aligned.jsonl (nb1) — bảo đảm cùng nguồn, cùng thứ tự
_test_aligned = load_jsonl(INPUT_FILES['test_aligned.jsonl'])
for name in ['test_run1', 'test_run2']:
    recs = PRED_SETS[name][0]
    assert len(recs) == len(_test_aligned), (name, len(recs), len(_test_aligned))
    n_mismatch = sum(1 for r, t in zip(recs, _test_aligned)
                     if r['text'] != t['text'] or r['corrected_text'] != t['corrected_text'])
    assert n_mismatch == 0, f'{name}: {n_mismatch} dòng lệch text/corrected_text vs test_aligned.jsonl'
print('Đối chiếu test_run{1,2} ↔ test_aligned.jsonl: KHỚP 100% (text + corrected_text, cùng thứ tự)')

val_recs, val_preds_r2 = PRED_SETS['val_run2']
mining_val_r2 = mine_fp_blocks(val_recs, val_preds_r2)

stats = mining_val_r2['stats']
print()
print('== Mining FP · VAL Run 2 (nguồn duy nhất của rule) ==')
print(f"  FP blocks: {stats['fp_blocks']} · mixed (không hoàn nguyên được): {stats['mixed_blocks_not_revertible']}")
print('  Theo type :', {k.replace('fp_type_', ''): v for k, v in sorted(stats.items()) if k.startswith('fp_type_')})
print(f"  punct-only: {stats['fp_punct_only']} block · top dấu: {mining_val_r2['punct_tokens'].most_common(5)}")
print(f"  Top-20 token bị xóa oan (→ R3): {mining_val_r2['deleted_tokens'].most_common(20)}")
print(f"  Top-10 phrase FP >=2 token (→ R4): {mining_val_r2['fp_phrases'].most_common(10)}")
print('  (Chiều hướng đối chiếu nb4 test: delete ~45,4% tổng FP — chỉ kiểm chứng, không derive từ test)')

Đối chiếu test_run{1,2} ↔ test_aligned.jsonl: KHỚP 100% (text + corrected_text, cùng thứ tự)

== Mining FP · VAL Run 2 (nguồn duy nhất của rule) ==
  FP blocks: 145 · mixed (không hoàn nguyên được): 784
  Theo type : {'delete': 105, 'merge': 5, 'multi': 2, 'split': 1, 'substitute': 32}
  punct-only: 21 block · top dấu: [('…', 10), (',', 5), (',…', 2), ('….', 1), (',...', 1)]
  Top-20 token bị xóa oan (→ R3): [('hòa', 10), ('thủy', 9), ('khỏe', 8), ('và', 8), (',', 8), ('thỏa', 7), ('tòa', 7), ('tùy', 5), ('khóa', 5), ('các', 4), ('xóa', 4), ('trong', 4), ('của', 4), ('họa', 3), ('việc', 2), ('thụy', 2), ('chức', 2), ('hóa', 2), ('ủy', 2), ('làm', 1)]
  Top-10 phrase FP >=2 token (→ R4): [('và làm việc', 1), ('trong việc', 1), ('trong kĩ năng ôn tập', 1), ('và kiểm soát', 1), ('cho đại lý cấp i quản lý', 1), ('của đề tài', 1), (', giám sát', 1), ('phễu cấp', 1), ('và các chức danh hợp nhất', 1), ('khi có tranh chấp', 1)]
  (Chiều hướng đối chiếu nb4 test: delete ~45,4% tổng FP — chỉ kiể

## 4. Sweep grid trên val Run 2 → chọn config

Grid **lồng nhau ≤6 tổ hợp** (chống overfit val n=927): G1 R1 → G2 +R2 → G3–G5 +R3 (k ∈ {1,3,5}) → G6 +R4. Baseline = raw (không post-processing).

Tiêu chí chọn (pre-registered): trong các config thỏa **over-correction val không tăng**, lấy **F1 val cao nhất**; chỉ nhận nếu **gain ≥ `MIN_VAL_GAIN` (0,3 điểm F1)**; hòa F1 → config đơn giản hơn (thứ tự trong grid). Không config nào đạt → **không áp post-processing** (config rỗng, ghi "không vượt ngưỡng hiệu quả").

In [7]:
GRID = [
    {'name': 'G1-r1',           'r1_punct_guard': True,  'r2_veto_valid_syllable': False, 'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': False},
    {'name': 'G2-r1r2',         'r1_punct_guard': True,  'r2_veto_valid_syllable': True,  'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': False},
    {'name': 'G3-r1r2-r3k1',    'r1_punct_guard': True,  'r2_veto_valid_syllable': True,  'r3_whitelist_min_count': 1, 'r4_phrase_blacklist': False},
    {'name': 'G4-r1r2-r3k3',    'r1_punct_guard': True,  'r2_veto_valid_syllable': True,  'r3_whitelist_min_count': 3, 'r4_phrase_blacklist': False},
    {'name': 'G5-r1r2-r3k5',    'r1_punct_guard': True,  'r2_veto_valid_syllable': True,  'r3_whitelist_min_count': 5, 'r4_phrase_blacklist': False},
    {'name': 'G6-r1r2-r3k5-r4', 'r1_punct_guard': True,  'r2_veto_valid_syllable': True,  'r3_whitelist_min_count': 5, 'r4_phrase_blacklist': True},
]
assert 1 <= len(GRID) <= 6, 'Grid phải <=6 tổ hợp (chống overfit val n=927)'
for _k in R3_K_GRID:
    assert any(c['r3_whitelist_min_count'] == _k for c in GRID), _k

raw_val_r2 = evaluate_predictions(val_recs, val_preds_r2, SYLL_SET)
baseline_f1 = raw_val_r2['detection']['f1']
baseline_overcorr = raw_val_r2['over_correction']['rate']

sweep_rows = []
for cfg in GRID:
    res = build_rule_resources(mining_val_r2, cfg['r3_whitelist_min_count'])
    post, post_stats = postprocess_preds(val_recs, val_preds_r2, cfg, res, SYLL_SET)
    m = evaluate_predictions(val_recs, post, SYLL_SET)
    sweep_rows.append({
        'config': cfg['name'], 'cfg': cfg,
        'f1': m['detection']['f1'], 'precision': m['detection']['precision'],
        'recall': m['detection']['recall'], 'overcorr': m['over_correction']['rate'],
        'clean_retention': m['over_correction']['clean_retention_rate'],
        'gain_f1': m['detection']['f1'] - baseline_f1,
        'n_changed': post_stats['n_changed_sentences'],
        'veto_by_rule': post_stats['veto_by_rule'],
    })

print(f"Baseline val Run 2 (raw): F1={baseline_f1:.2%} · over-corr={baseline_overcorr:.2%}")
print(f"  {'config':<18s} | {'dF1':>8s} | {'F1':>7s} | {'Prec':>7s} | {'Rec':>7s} | {'Over-corr':>9s} | {'#cau doi':>8s} | hit rule")
for row in sweep_rows:
    print(f"  {row['config']:<18s} | {row['gain_f1']:>+8.2%} | {row['f1']:>7.2%} | {row['precision']:>7.2%} "
          f"| {row['recall']:>7.2%} | {row['overcorr']:>9.2%} | {row['n_changed']:>8d} | {row['veto_by_rule']}")

eligible = [r for r in sweep_rows
            if r['overcorr'] <= baseline_overcorr and r['gain_f1'] >= MIN_VAL_GAIN]
_GRID_IDX = {c['name']: i for i, c in enumerate(GRID)}
if eligible:
    best = max(eligible, key=lambda r: (r['f1'], -_GRID_IDX[r['config']]))
    assert best['gain_f1'] >= MIN_VAL_GAIN
    CHOSEN_CFG = dict(best['cfg'])
    del CHOSEN_CFG['name']
    CHOSEN_NAME = best['config']
    print(f">> CHON: {CHOSEN_NAME} · gain dF1 val = {best['gain_f1']:+.2%} · "
          f"over-corr val {best['overcorr']:.2%} (baseline {baseline_overcorr:.2%})")
else:
    CHOSEN_CFG = {'r1_punct_guard': False, 'r2_veto_valid_syllable': False,
                  'r3_whitelist_min_count': 0, 'r4_phrase_blacklist': False}
    CHOSEN_NAME = 'EMPTY-khong-vuot-nguong'
    print(f">> Khong config nao dat nguong hieu qua (gain >= {MIN_VAL_GAIN:.1%} F1 voi over-correction khong tang) "
          '— post-processing KHONG AP (config rong).')

CHOSEN_APPLIED = any([CHOSEN_CFG['r1_punct_guard'], CHOSEN_CFG['r2_veto_valid_syllable'],
                      CHOSEN_CFG['r3_whitelist_min_count'] > 0, CHOSEN_CFG['r4_phrase_blacklist']])
CHOSEN_RES = build_rule_resources(mining_val_r2, CHOSEN_CFG['r3_whitelist_min_count'])

Baseline val Run 2 (raw): F1=74.67% · over-corr=0.81%
  config             |      dF1 |      F1 |    Prec |     Rec | Over-corr | #cau doi | hit rule
  G1-r1              |   +0.67% |  75.34% |  80.64% |  70.69% |     0.73% |       20 | {'R1': 22}
  G2-r1r2            |   +4.13% |  78.80% |  92.25% |  68.77% |     0.25% |      119 | {'R2': 117, 'R1': 22}
  G3-r1r2-r3k1       |   +4.69% |  79.35% |  93.79% |  68.77% |     0.20% |      121 | {'R2': 117, 'R1': 22, 'R3': 5}
  G4-r1r2-r3k3       |   +4.13% |  78.80% |  92.25% |  68.77% |     0.25% |      119 | {'R2': 117, 'R1': 22}
  G5-r1r2-r3k5       |   +4.13% |  78.80% |  92.25% |  68.77% |     0.25% |      119 | {'R2': 117, 'R1': 22}
  G6-r1r2-r3k5-r4    |   +4.13% |  78.80% |  92.25% |  68.77% |     0.25% |      119 | {'R2': 117, 'R1': 22}
>> CHON: G3-r1r2-r3k1 · gain dF1 val = +4.69% · over-corr val 0.20% (baseline 0.81%)


## 5. Frozen áp lên val (báo cáo) + test Run 1/Run 2

Config + resources (whitelist/blacklist derive từ **val**) đóng băng → áp cho 4 prediction set. Test chưa từng tham gia derive rule; **Run 1 test = robustness check** (config tune trên Run 2 val có tổng quát sang model thuần không). Đối chiếu ΔF1 test Run 2 với trần lý thuyết **+2,33 điểm** (nb4) — không kỳ vọng đạt trần vì rule chỉ chặn được phần delete-FP có pattern.

In [8]:
frozen_results = {}
POST_PREDS = {}
print(f"Frozen config: {CHOSEN_NAME} = {CHOSEN_CFG} (applied={CHOSEN_APPLIED})")
print(f"  {'set':<10s} | {'dPrec':>8s} | {'dRec':>8s} | {'dF1':>8s} | {'dOver-corr':>10s} | {'dCleanRet':>9s} | {'#doi':>5s} | hit rule")
for name in ['val_run1', 'val_run2', 'test_run1', 'test_run2']:
    recs, preds = PRED_SETS[name]
    raw = evaluate_predictions(recs, preds, SYLL_SET)
    if CHOSEN_APPLIED:
        post, post_stats = postprocess_preds(recs, preds, CHOSEN_CFG, CHOSEN_RES, SYLL_SET)
    else:
        post, post_stats = list(preds), {'n_changed_sentences': 0, 'veto_by_rule': {}}
    m = evaluate_predictions(recs, post, SYLL_SET)
    frozen_results[name] = {
        'raw': raw, 'post': m,
        'delta': {
            'precision': m['detection']['precision'] - raw['detection']['precision'],
            'recall': m['detection']['recall'] - raw['detection']['recall'],
            'f1': m['detection']['f1'] - raw['detection']['f1'],
            'overcorr': m['over_correction']['rate'] - raw['over_correction']['rate'],
            'clean_retention': m['over_correction']['clean_retention_rate'] - raw['over_correction']['clean_retention_rate'],
        },
        'postproc': post_stats,
    }
    POST_PREDS[name] = {'preds': post, 'stats': post_stats}
    d = frozen_results[name]['delta']
    print(f"  {name:<10s} | {d['precision']:>+8.2%} | {d['recall']:>+8.2%} | {d['f1']:>+8.2%} "
          f"| {d['overcorr']:>+10.2%} | {d['clean_retention']:>+9.2%} | {post_stats['n_changed_sentences']:>5d} "
          f"| {post_stats['veto_by_rule']}")

_t2 = frozen_results['test_run2']
print()
print(f"Doi chieu tran: dF1 test Run 2 = {_t2['delta']['f1']:+.2%} vs tran ly thuyet "
      f"+{THEORETICAL_CEILING_F1_GAIN:.2%} (chan het delete-FP — nb4) · over-correction test "
      f"{'KHONG tang' if _t2['delta']['overcorr'] <= 0 else 'TANG — ra lai!'}")
print('Run 1 test = robustness check: config tune tren val Run 2 ap sang model thuan (Run 1).')

Frozen config: G3-r1r2-r3k1 = {'r1_punct_guard': True, 'r2_veto_valid_syllable': True, 'r3_whitelist_min_count': 1, 'r4_phrase_blacklist': False} (applied=True)
  set        |    dPrec |     dRec |      dF1 | dOver-corr | dCleanRet |  #doi | hit rule
  val_run1   |  +13.61% |   -2.27% |   +4.12% |     -0.56% |   +33.33% |   116 | {'R2': 114, 'R1': 18, 'R3': 3}
  val_run2   |  +14.79% |   -2.01% |   +4.69% |     -0.61% |   +33.33% |   121 | {'R2': 117, 'R1': 22, 'R3': 5}
  test_run1  |   +2.72% |   -0.35% |   +0.83% |     -0.41% |   +10.11% |   524 | {'R2': 550, 'R1': 58, 'R3': 1}
  test_run2  |   +2.63% |   -0.32% |   +0.93% |     -0.42% |   +10.57% |   551 | {'R2': 549, 'R1': 81, 'R3': 1}

Doi chieu tran: dF1 test Run 2 = +0.93% vs tran ly thuyet +2.33% (chan het delete-FP — nb4) · over-correction test KHONG tang
Run 1 test = robustness check: config tune tren val Run 2 ap sang model thuan (Run 1).


## 6. QA + kiểm định bất biến

- 10 mẫu đầu **bị thay đổi** trên test Run 2: src / gold / pred gốc / pred sau rule (kèm rule đã bắn).
- Kiểm định mọi output: NFC chuẩn (assert trong `postprocess_one`) + roundtrip token hóa (tokenize ngược khớp danh sách token đã rebuild) — 4/4 set.

In [9]:
# Kiểm định NFC 4/4 set (assert NFC + roundtrip đã nằm trong postprocess_one cho bản rebuild)
for name in ['val_run1', 'val_run2', 'test_run1', 'test_run2']:
    for _t in POST_PREDS[name]['preds']:
        assert unicodedata.is_normalized('NFC', _t), name
print('NFC PASS trên 4/4 set output post-processing')

qa_samples = []
qa_shown = 0
recs_t2, raw_t2 = PRED_SETS['test_run2']
post_t2 = POST_PREDS['test_run2']['preds']
print()
print('=== QA · test Run 2 — 10 mẫu đầu bị thay đổi bởi rule ===')
for rec, p_raw, p_post in zip(recs_t2, raw_t2, post_t2):
    if p_raw == p_post or qa_shown >= 10:
        continue
    _src = canon_tokenize(rec['text'])
    _blocks = extract_edit_blocks(_src, canon_tokenize(p_raw), levenshtein_opcodes(_src, canon_tokenize(p_raw)))
    fired = [veto_reason(b, CHOSEN_CFG, CHOSEN_RES, SYLL_SET) for b in _blocks]
    fired = [f for f in fired if f]
    print(f"[{rec.get('row_id')}] rule fired: {fired}")
    print('  SRC       :', rec['text'])
    print('  GOLD      :', rec['corrected_text'])
    print('  PRED raw  :', p_raw)
    print('  PRED post :', p_post)
    print()
    qa_samples.append({'row_id': rec.get('row_id'), 'src': rec['text'], 'gold': rec['corrected_text'],
                       'pred_raw': p_raw, 'pred_post': p_post, 'rules_fired': fired})
    qa_shown += 1
if qa_shown == 0:
    print('(Khong co mau nao thay doi — config rong hoac rule khong ban tren test Run 2)')

NFC PASS trên 4/4 set output post-processing

=== QA · test Run 2 — 10 mẫu đầu bị thay đổi bởi rule ===
[None] rule fired: ['R2', 'R2']
  SRC       : Đây là nhà máy thủy điện cuối cùng chong các bậc than thủy điện chên sông Sê San.
  GOLD      : Đây là nhà máy thủy điện cuối cùng trong các bậc thang thủy điện trên sông Sê San.
  PRED raw  : Đây là nhà máy điện cuối cùng chong các bậc than điện chên sông Sê San.
  PRED post : Đây là nhà máy thủy điện cuối cùng chong các bậc than thủy điện chên sông Sê San .

[None] rule fired: ['R2']
  SRC       : Hệ thống trính sách thuế phải tạo điều ciện thúc đẩy cải cách hàn trính thuế theo hướng vừa đáp ứng yêu cầu quản lý thuế của Nhà nước, vừa không gây phiền hà, tốn cém tro cả tổ trức, cá nhân nộp thuế và cơ quan thuế.
  GOLD      : Hệ thống chính sách thuế phải tạo điều kiện thúc đẩy cải cách hành chính thuế theo hướng vừa đáp ứng yêu cầu quản lý thuế của Nhà nước, vừa không gây phiền hà, tốn kém cho cả tổ chức, cá nhân nộp thuế và cơ quan thuế

## 7. Xuất file

- `postprocess_config.json` — config frozen + resources (whitelist/blacklist val) — **nb7 nạp để đối chiếu, re-tune lại trên val Run 3**.
- `postprocess_report.json` — mining val, sweep, kết quả raw/post/Δ từng set, QA samples.
- `predictions_{val,test}_run{1,2}_postproc.jsonl` — schema nb3 + `prediction` (sau rule) + `prediction_raw` (trước rule).

In [10]:
OUT_FILES = ['postprocess_config.json', 'postprocess_report.json']
for name in ['val_run1', 'val_run2', 'test_run1', 'test_run2']:
    recs, raw_preds = PRED_SETS[name]
    post_preds = POST_PREDS[name]['preds']
    out_path = OUTPUT_DIR / f'predictions_{name}_postproc.jsonl'
    with open(out_path, 'w', encoding='utf-8') as f:
        for r, p_raw, p_post in zip(recs, raw_preds, post_preds):
            f.write(json.dumps({'row_id': r.get('row_id'), 'text': r['text'],
                                'corrected_text': r['corrected_text'],
                                'prediction': p_post, 'prediction_raw': p_raw}, ensure_ascii=False) + '\n')
    OUT_FILES.append(out_path.name)

chosen_whitelist = sorted(
    (t, n) for t, n in mining_val_r2['deleted_tokens'].items()
    if CHOSEN_CFG['r3_whitelist_min_count'] > 0 and n >= CHOSEN_CFG['r3_whitelist_min_count'])
chosen_phrases = sorted(mining_val_r2['fp_phrases'].items()) if CHOSEN_CFG['r4_phrase_blacklist'] else []

postproc_config = {
    'created': RUN_STAMP,
    'notebook': 'nb6_post_processing',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'seed': SEED,
    'min_val_gain': MIN_VAL_GAIN,
    'grid': [c['name'] for c in GRID],
    'chosen_config_name': CHOSEN_NAME,
    'chosen_config': CHOSEN_CFG,
    'applied': CHOSEN_APPLIED,
    'resources': {
        'source': 'val_run2_fp_mining_only (DESIGN.md §9 — khong derive tu test)',
        'phrase_min_count': PHRASE_MIN_COUNT,
        'syllable_table_n': len(SYLL_SET),
        'r3_whitelist': chosen_whitelist,
        'r4_phrase_blacklist': chosen_phrases,
    },
    'val_run2_baseline': {'f1': baseline_f1, 'overcorr': baseline_overcorr},
    'val_run2_chosen': {k: v for k, v in next(
        (r for r in sweep_rows if r['config'] == CHOSEN_NAME), {'f1': baseline_f1, 'gain_f1': 0.0}).items()
        if k in ('f1', 'gain_f1', 'overcorr')},
    'notebook_next': 'nb7 nap file nay de doi chieu; re-tune cung rule-shape tren val Run 3 (khong ap lai config Run 2)',
}
with open(OUTPUT_DIR / 'postprocess_config.json', 'w', encoding='utf-8') as f:
    json.dump(postproc_config, f, ensure_ascii=False, indent=2)

postprocess_report = {
    'created': RUN_STAMP,
    'notebook': 'nb6_post_processing',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'leakage_note': 'Rule + whitelist + blacklist derive CHI tu val Run 2; test chi duoc ap config frozen '
                    '(Run 1 test = robustness check). Findings nb4 tren test khong tham gia derive.',
    'config': {**CHOSEN_CFG, 'name': CHOSEN_NAME, 'applied': CHOSEN_APPLIED,
               'min_val_gain': MIN_VAL_GAIN, 'phrase_min_count': PHRASE_MIN_COUNT,
               'grid': [c['name'] for c in GRID],
               'theoretical_ceiling_f1_gain_test_run2': THEORETICAL_CEILING_F1_GAIN},
    'mining_val_run2': {
        'stats': dict(mining_val_r2['stats']),
        'top_deleted_tokens': mining_val_r2['deleted_tokens'].most_common(30),
        'top_fp_phrases': mining_val_r2['fp_phrases'].most_common(15),
        'top_punct_tokens': mining_val_r2['punct_tokens'].most_common(10),
    },
    'baseline_val_run2': {'f1': baseline_f1, 'overcorr': baseline_overcorr},
    'sweep_val_run2': [{k: row[k] for k in ['config', 'f1', 'precision', 'recall', 'overcorr',
                                            'clean_retention', 'gain_f1', 'n_changed', 'veto_by_rule']}
                       for row in sweep_rows],
    'selection': {'eligible': [r['config'] for r in eligible], 'chosen': CHOSEN_NAME},
    'results_frozen': frozen_results,
    'qa_samples_changed_test_run2': qa_samples,
    'files': OUT_FILES,
}
with open(OUTPUT_DIR / 'postprocess_report.json', 'w', encoding='utf-8') as f:
    json.dump(postprocess_report, f, ensure_ascii=False, indent=2)

print('=== HOÀN TẤT nb6 ===')
print('Đã ghi vào', OUTPUT_DIR)
for fn in OUT_FILES:
    print(' -', fn)

=== HOÀN TẤT nb6 ===
Đã ghi vào /kaggle/working
 - postprocess_config.json
 - postprocess_report.json
 - predictions_val_run1_postproc.jsonl
 - predictions_val_run2_postproc.jsonl
 - predictions_test_run1_postproc.jsonl
 - predictions_test_run2_postproc.jsonl
